In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns
from scipy import stats

In [2]:
df = pd.read_csv('../data/cleaned/India.csv')

In [3]:
indicators = {
    "vaccination": "Children age 12-23 months fully vaccinated based on information from either vaccination card or mother's recall (%)",
    "maternal_health": "Mothers who had at least 4 antenatal care visits (%)",
    "malnutrition": "Children under 5 years who are stunted (height for age) (%)",
}

vacc_df = df[df['Indicator'] == indicators['vaccination']]
matr_df = df[df['Indicator'] == indicators['maternal_health']]
mal_df  = df[df['Indicator'] == indicators['malnutrition']]

In [4]:
northeast_df = vacc_df[vacc_df['State'].isin(['Arunachal Pradesh', 'Assam', 'Manipur', 'Meghalaya','Mizoram', 'Nagaland', 'Sikkim', 'Tripura'])]

In [5]:
mainland_df = vacc_df[~vacc_df['State'].isin(['Arunachal Pradesh', 'Assam', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Sikkim', 'Tripura'])]

Northeast states vs National average

In [6]:
stats.ttest_ind(
    northeast_df['NFHS_5'],
    mainland_df['NFHS_5'],
    equal_var=False,
)

TtestResult(statistic=np.float64(-7.11142938040048), pvalue=np.float64(4.261673439631968e-11), df=np.float64(151.2932007327219))

H0 : There is no signficant difference between Northeast states and other states regarding vaccinaton rates<br>
H1 : Northeast vaccination rates are significantly lower than the national average<br>
Test : One-tailed independent t-test<br>
Significane level : 0.05<br>

In [7]:
health_df = pd.read_csv('../data/cleaned/health_merged.csv')

In [8]:
mal_mean = health_df['child malnutrition'].mean()

In [9]:
high_mal = health_df[health_df['child malnutrition'] > mal_mean]
low_mal = health_df[health_df['child malnutrition'] < mal_mean]

Malnutrition and Vaccination

In [10]:
stats.ttest_ind(high_mal['vaccination rates'], low_mal['vaccination rates'], equal_var=False)

TtestResult(statistic=np.float64(-4.711745172987805), pvalue=np.float64(2.9854678884121315e-06), df=np.float64(672.6709083180658))

H0 : There is no significant difference in vaccination for high and low malnutrition districts<br>
H1 : Districts with high vaccination have significantly lower vaccination and vice versa<br>
Test : One-tailed independent t-test<br>
Significane level : 0.05<br>

In [11]:
m5at_df = matr_df['NFHS_5']
m4at_df = matr_df['NFHS_4']
stats.ttest_rel(m5at_df, m4at_df, alternative='greater')

TtestResult(statistic=np.float64(19.081207229029587), pvalue=np.float64(4.316217577015989e-66), df=np.int64(707))

H0 : There is no significant difference in maternal health for NFHS_5 and NFHS_4<br>
H1 : Maternal health improved significantly between NFHS_4 and NFHS_5<br>
Test : One-tailed paired test<br>
Significane : 0.05<br>


In [12]:
matr_df['change'] = matr_df['NFHS_5'] - matr_df['NFHS_4']

In [13]:
stats.t.interval(confidence=0.95, df=len(matr_df)-1, loc=matr_df['change'].mean(), scale=stats.sem(matr_df['change']))

(np.float64(16.906283160668885), np.float64(20.78439480543281))

In [14]:
national_mean = health_df['vaccination rates'].mean()
health_df['vacc_category'] = health_df['vaccination rates'].apply(lambda x: 'High' if x > national_mean else 'Low')

In [15]:
northeast_states = ['Arunachal Pradesh', 'Assam', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Sikkim', 'Tripura']
health_df['region'] = health_df['State'].apply(lambda x: 'Northeast' if x in northeast_states else 'Mainland')

In [16]:
contingency_table = pd.crosstab(health_df['region'], health_df['vacc_category'])

In [17]:
stats.chi2_contingency(contingency_table)

Chi2ContingencyResult(statistic=np.float64(49.121022463832354), pvalue=np.float64(2.40647247571092e-12), dof=1, expected_freq=array([[292.74264706, 284.25735294],
       [ 52.25735294,  50.74264706]]))

H0 : Region and vaccination performance are independent<br>
H1 : Region and vaccination are significantly related<br>
Test : Chi-squared test of independence<br>
Significane level : 0.05<br>

In [18]:
vacc_change = vacc_df['NFHS_5'].mean() - vacc_df['NFHS_4'].mean()
matr_change = matr_df['NFHS_5'].mean() - matr_df['NFHS_4'].mean()
mal_change = mal_df['NFHS_5'].mean() - mal_df['NFHS_4'].mean()
print(vacc_change)
print(matr_change)
print(mal_change)

25.555225988700563
18.845338983050844
4.15112994350282


While vaccination and maternal health improved significantly between 2015-16 and 2019-21, 
child malnutrition worsened nationally by 4.15 percentage points on average. 
This decline occurred despite the expansion of national nutrition programs during the same period, 
suggesting that healthcare access and child nutrition are driven by different underlying factors.